# 2. VAE pools (TensorFlow + tsgm)

> **Runtime: GPU.** Runtime → Change runtime type → **GPU** (T4 or better). TensorFlow trains the VAE; on CPU it is many times slower and the times in RQ6 are not comparable.

This notebook has its own Colab runtime, and that runtime is the tsgm environment: `tsgm` is installed on top of Colab's TensorFlow (GPU) and Keras 3, and the `pools` stage runs in a fresh subprocess of this runtime (TensorFlow is loaded there before anything touches CUDA). The pools and VAE weights go to the artifacts folder on Drive; notebook 3 only reads them and never imports TensorFlow.

**Order:** 1 → 2 → 3 → 4 → 5 → 6 → 7. When this notebook has finished, close it with Runtime → *Disconnect and delete runtime* (frees the GPU), then open notebook 3 and run it on a **GPU** runtime as well. Every notebook starts its own runtime; nothing has to be carried over except the files on Drive.

In [ ]:
# ── Drive first: the project (and colab_setup.py) lives there ──
from google.colab import drive
drive.mount("/content/drive")

# ── setup: Drive, code, paths ──
REPO = "/content/drive/MyDrive/Colab Notebooks/MarathonTS"   # project folder on Drive (or a git clone)
PROFILE = "min"            # "min": the Colab run (fast); "mid": min + the VAE per seed; "max": everything on
MAIN_ENVS = ["base"]       # this runtime needs no torch stack: only tsgm on top of Colab's TensorFlow
# results go to {REPO}/artifacts/<PROFILE>; set ARTIFACTS = "..." to change
exec(open(f"{REPO}/colab/colab_setup.py").read())
install_tsgm()             # tsgm 0.1.0 (patched setup.py for Python 3.13), once per runtime
import subprocess, sys
_chk = subprocess.run([sys.executable, "-c", "import os; os.environ['KERAS_BACKEND']='tensorflow'; "
                       "import tensorflow as tf, keras, tsgm; print('tf', tf.__version__, '| keras', keras.__version__, "
                       "'| tsgm ok | GPUs:', len(tf.config.list_physical_devices('GPU')))"],
                      capture_output=True, text=True)
print(_chk.stdout or _chk.stderr[-3000:])
assert _chk.returncode == 0, "TensorFlow / tsgm do not import in this runtime — see the error above"

In [ ]:
# ── runtime check: this notebook expects a GPU ──
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", out.stdout.strip() if out.returncode == 0 else "NONE — Runtime → Change runtime type → GPU")

In [ ]:
# ── scope of this run: change freely; finished units are skipped on re-run ──
import dataclasses
cfg = dataclasses.replace(
    cfg,
    datasets=list(cfg.datasets),  # the profile's datasets (marathonts/bench/config.py DATASETS); or a list, e.g. ["ett", "my_ds"]
    methods=None,                # None → the suites on every pool + long_methods; or a list to run a subset
    baselines=None,              # None → long_methods (case study: CS9, CS10)
    # custom components (EXTENDING.md): plugin files + one line each, no code changes:
    # plugins=["my_plugins.py"],                          # files registering them (path from the project folder)
    # pool_suites={"vae": "tsgm_vae", "gan": "my_gan"},   # the whole suite on another pool generator → gan_zseries, …
    # extra_assemblers=["my_stitch"],                     # a stitcher on every pool → vae_my_stitch, …
    # extra_joiners=["my_smooth"],                        # your own seam processing (run like the built-in ones)
    # smooth_paths=["zseries", "greedy"],                 # stitchers that get seam processing ("all": all but the random path and vae_quilting; min: ["zseries"])
    # joiners=["crossfade", "wsola"],                     # which seam processings (None: all)
    # disabled=["pctsp", "chronos"],                      # switch off methods or components by name
    # noise_twin=True,                                    # + the noise twin vae_rand_b (min: off; mid, max: on)
    # frozen_methods=[],                                  # published results kept as they are, not run (default: vae_zseries and its
    #                                                     #   seam-processing variants, marathonts/bench/config.py FROZEN_METHODS); [] → left out of the tables
    # long_methods=[*cfg.long_methods, "my_long"],          # + a long-series generator (method and baseline)
    seeds=list(range(10)),
)
# plugins=[...] above loads custom components (datasets, generators, assemblers, joiners, models, metrics, tasks) in
# every notebook, in the stage subprocesses and in the hypothesis part — EXTENDING.md

In [ ]:
# tsgm VAE pools: `pools` stage in a fresh process of this runtime (TensorFlow + tsgm, no torch import before TF)
run_in_env(None, "pools", "--pool-generators", "tsgm_vae", config=cfg)   # the whole cfg goes to the subprocess

In [ ]:
# another pool generator in its own environment, e.g. SDV:
# run_in_env("sdv", "pools", "--pool-generators", "sdv_par", config=cfg)
# custom pool generators: cfg.plugins are loaded in the subprocess too; a generator that needs no
# separate environment is also trained in notebook 3 (stage generate computes missing pools)

In [ ]:
import pandas as pd, glob
from marathonts.bench.storage import read_record
rows = [dict(read_record(f), file=f.split("/pools/")[1]) for f in glob.glob(f"{cfg.artifacts_dir}/pools/*/*/seed*.record.parquet")]
if rows:
    df = pd.DataFrame(rows)
    display(df[["file", "fit_wall_s", "sample_wall_s"] + [c for c in ("vae_epochs_run", "vae_holdout_loss", "tf_gpus", "pool_gpu") if c in df]])   # tf_gpus = 0 → TensorFlow ran on CPU
else:
    print("no pools yet — see the log above and", f"{cfg.artifacts_dir}/errors.log")